In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [ ]:
from pathlib import Path

import torch
import torch.multiprocessing as mp
import tqdm
from jppype import vscode_theme
from torch import nn
from torch_geometric.loader import DataLoader
from torch_geometric.transforms import ToDevice

from fundus_vessels_toolkit.segment_to_graph.models.dataset import VBranchDigraphDataset

vscode_theme()

In [ ]:
# torch.backends.fp32_precision = "tf32"

In [ ]:
PATH = [
    Path("/run/media/gaby/GREY SSD/PostDoc/DATA/Fundus/" + folder)
    for folder in ["GAVE-train", "MAPLES-DR", "Fundus-AV"]
]
RAW = [path / "1-images" for path in PATH]
AV = [path / "2-av-pred_CLEMENT" for path in PATH]
TOPO = [path / "3-topo" for path in PATH]

In [ ]:
VBranchDigraphDataset.load_from_dirs(RAW, TOPO, av_dir=AV, resize_to=1024)

In [ ]:
dataset = VBranchDigraphDataset.load_from_dirs(RAW, TOPO, av_dir=AV, resize_to=1024)
dataset.augment = True

In [ ]:
m, digraph, fundus_img, od_yx, mac_yx = dataset.draw_jppype(
    "g_022", augment=False, test=True, branch_label=True, node_label=True
)
m

In [ ]:
digraph.graph.branch_list[166]

In [ ]:
from fundus_vessels_toolkit.segment_to_graph.models.digraph_model import (
    BranchDigraphModel,
    BranchFeaturesEfficientNetV2S,
    Gatv2GCN,
)
import torch.nn.functional as F

torch._dynamo.config.capture_dynamic_output_shape_ops = True
model = BranchDigraphModel(BranchFeaturesEfficientNetV2S(), Gatv2GCN(n_in=784, n_out=512))
model = model.cuda()
# model = torch.compile(model, dynamic=True)


In [ ]:
av_nll_loss = nn.NLLLoss()
dir_bce_loss = nn.BCEWithLogitsLoss()
line_ce_loss = nn.CrossEntropyLoss()

for i, batch in enumerate(tqdm.tqdm(DataLoader(dataset, batch_size=3, num_workers=5))):
    av_p, dir_p, edge_p, valid_lines, valid_roots = model(batch.cuda())
    # AV loss
    target_av_p = torch.cat([1 - batch.branch_av_p.sum(dim=-1, keepdim=True), batch.branch_av_p], dim=-1)
    target_av = torch.argmax(target_av_p, dim=-1)
    av_loss = av_nll_loss(F.log_softmax(av_p, dim=1), target_av)

    # Dir and line losses
    dir_loss = dir_bce_loss(dir_p, batch.branch_dir)
    edge_p_gt = torch.cat([batch.edge_p[valid_lines], batch.branch_root_p[valid_roots]], dim=0)
    line_loss = line_ce_loss(edge_p, edge_p_gt)

    loss = av_loss + dir_loss + line_loss
    loss.backward()